# Convolutional Neural Networks — CIFAR-10

This chapter is the first hands-on **CNN / Computer Vision** project in the `AI_From_Scratch` learning path.

The goal is not to maximize the CIFAR-10 leaderboard. The goal is to understand the complete CNN workflow by building it, training it, evaluating it, and running controlled experiments.

## Learning objectives

- `Conv2d`, filters, channels, and feature maps
- Kernel size, stride, and padding
- ReLU activation
- Max pooling
- Flattening convolutional features
- Fully connected classification
- `CrossEntropyLoss`
- Adam optimization
- Epoch-based training loops
- PyTorch XPU acceleration
- Training vs. test accuracy
- Confusion matrices
- Model capacity and generalization
- Data augmentation
- Visual inspection of mistakes


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.optim import Adam
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

## 1. Device selection

The training environment uses Intel GPU acceleration through PyTorch's **XPU** backend when available, with CPU as a fallback.


In [ ]:
if torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

print(device)

## 2. CIFAR-10 dataset

CIFAR-10 contains:

- 50,000 training images
- 10,000 test images
- RGB images
- Image size: `32 × 32`
- 10 classes

### Class mapping

| Index | Class |
|---:|---|
| 0 | airplane |
| 1 | automobile |
| 2 | bird |
| 3 | cat |
| 4 | deer |
| 5 | dog |
| 6 | frog |
| 7 | horse |
| 8 | ship |
| 9 | truck |

For training, random horizontal flips and random crops are used as data augmentation. The test set is only converted to tensors.


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.CIFAR10(
    root="/data/Datasets/Images/First_CNN_CIFAR10_Dataset/",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root="/data/Datasets/Images/First_CNN_CIFAR10_Dataset/",
    train=False,
    download=True,
    transform=test_transform
)

## 3. DataLoaders

Training uses batches of 64 and shuffles the training data. Test data is not shuffled.


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

## 4. CNN architecture

The final model used in the experiments is:

```text
Input: 3 × 32 × 32
        ↓
Conv2d: 3 → 32, 3×3, padding=1
        ↓
ReLU
        ↓
MaxPool 2×2
        ↓
Conv2d: 32 → 64, 3×3, padding=1
        ↓
ReLU
        ↓
MaxPool 2×2
        ↓
Flatten: 64 × 8 × 8 = 4096
        ↓
Linear: 4096 → 10
```

The two pooling operations reduce the spatial dimensions:

```text
32 × 32 → 16 × 16 → 8 × 8
```

The final convolution produces 64 feature maps, so:

```text
64 × 8 × 8 = 4096
```

features enter the classifier.


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(4096, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

## 5. Initialize the model

The model is moved to the selected device before training.


In [ ]:
model = CNN()
model.to(device)

print(model)

## 6. Loss function and optimizer

This is a 10-class classification problem.

- Loss: `CrossEntropyLoss`
- Optimizer: Adam
- Learning rate: `0.001`

The CNN produces raw logits, which are passed directly to `CrossEntropyLoss`.


In [ ]:
loss_fn = nn.CrossEntropyLoss()

optimizer = Adam(
    params=model.parameters(),
    lr=0.001
)

## 7. Training loop

For every batch:

```text
zero gradients
      ↓
forward pass
      ↓
calculate loss
      ↓
backpropagation
      ↓
update parameters
```

The loop below tracks both average training loss and training accuracy.


In [ ]:
epochs = 20
model.train()

for epoch in range(epochs):
    total_loss = 0
    correct = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_fn(outputs, labels)

        total_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()

        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / len(train_dataset) * 100

    print(
        f"Epoch: {epoch} | "
        f"Loss: {avg_loss:.8f} | "
        f"Correct: {correct} | "
        f"Accuracy: {accuracy:.2f}%"
    )

## 8. Test evaluation

Training accuracy only tells us how well the model fits the training data.

We therefore evaluate on the separate test set using:

- `model.eval()`
- `torch.no_grad()`

We also save every prediction and true label so that a confusion matrix can be calculated.


In [ ]:
model.eval()

with torch.no_grad():
    correct = 0
    predictions = []
    actuals = []

    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predicted = outputs.argmax(dim=1)

        correct += (predicted == labels).sum().item()

        predictions.append(predicted)
        actuals.append(labels)

    print(f"Accuracy: {correct / len(test_dataset) * 100:.2f}%")

    predictions = torch.cat(predictions)
    actuals = torch.cat(actuals)

## 9. Confusion matrix

A confusion matrix gives more information than one accuracy number.

- **Rows** = actual class
- **Columns** = predicted class
- **Diagonal** = correct predictions
- **Off-diagonal cells** = specific mistakes

This helps identify which classes the CNN tends to confuse.


In [ ]:
cm = confusion_matrix(
    actuals.to("cpu"),
    predictions.to("cpu")
)

print(cm)

## 10. Inspecting predictions

We can inspect a test batch visually. CIFAR-10 images are only `32 × 32`, so some examples are genuinely difficult to distinguish.

First we obtain one batch and its predictions.


In [ ]:
model.eval()

with torch.no_grad():
    images, labels = next(iter(test_loader))

    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    predicted = outputs.argmax(dim=1)

print(images.shape)
print(labels.shape)
print(predicted.shape)

## 11. Find misclassified images

We compare predictions with the true labels and find the indices where they differ.


In [ ]:
wrong_indices = torch.where(predicted != labels)[0]

print(wrong_indices)

## 12. Visual inspection of a mistake

Looking at actual failures connects the numerical evaluation to the images the model finds difficult.


In [ ]:
if len(wrong_indices) > 0:
    index = wrong_indices[0]

    plt.figure(figsize=(4, 4))
    plt.imshow(images[index].permute(1, 2, 0).cpu())
    plt.axis("off")
    plt.show()

    print("Actual:", labels[index].item())
    print("Predicted:", predicted[index].item())
else:
    print("No misclassified images in this batch.")

# Experiments and observations

The experiments were designed to understand **generalization**, rather than to exhaustively optimize CIFAR-10.

## Experiment 1 — Train longer

The smaller model with 16 → 32 channels was trained for 10 and 20 epochs.

| Experiment | Training accuracy | Test accuracy |
|---|---:|---:|
| 16 → 32, 10 epochs | 70.72% | 66.70% |
| 16 → 32, 20 epochs | 73.74% | 68.63% |

Training longer improved both training and test performance in this experiment.

## Experiment 2 — Increase model capacity

The convolution channels were increased from 16 → 32 to 32 → 64.

| Experiment | Training accuracy | Test accuracy |
|---|---:|---:|
| 32 → 64, 20 epochs | 81.92% | 70.27% |

The larger model fitted the training set substantially better, while the improvement on unseen data was smaller.

## Experiment 3 — Data augmentation

Adding a random horizontal flip reduced training accuracy while improving test accuracy.

Adding a random crop produced another improvement:

| Experiment | Training accuracy | Test accuracy |
|---|---:|---:|
| 32 → 64 + horizontal flip | 76.98% | 72.24% |
| 32 → 64 + flip + random crop | 71.05% | **73.80%** |

This demonstrated an important distinction between **fitting** and **generalization**.

The final measured test accuracy was **73.80%**.

## Key takeaway

A model can become harder to fit on the training data while becoming better at generalizing to unseen data.

The purpose of this chapter was to understand the CNN pipeline and these concepts, not to spend excessive time optimizing a small CIFAR-10 model.

## Next topic

**Sequence Models → RNNs → LSTMs/GRUs → Attention → Transformers**
